# TP5 — Test A/B en Python

**Bootcamp Cybersécurité, Pentest & AB Testing — YNOV Montpellier**

Objectif : mener un test A/B de bout en bout et interpréter le résultat.
Reportez vos résultats dans le **Cahier de TP** au fur et à mesure.

> 🔑 Rappel : A = version *contrôle*, B = version *variante*. On teste si B change vraiment le taux de conversion.

---
> ✏️ **Version à compléter** — remplissez les lignes marquées `# TODO` et les `____`.
> Appuyez-vous sur le support de cours (Module 3) ; en cas de blocage, sollicitez le formateur.

## Étape 1 — Générer un jeu de données simulé
Deux groupes de 5 000 visiteurs. Groupe A : ~10 % de conversion. Groupe B : ~12 %.
`np.random.seed(42)` fige le hasard pour que tout le monde obtienne les mêmes chiffres.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)
n_a, n_b = 5000, 5000
# Groupe A (contrôle) : taux de conversion ~10 %
groupe_a = np.random.binomial(1, 0.10, n_a)
# TODO : créez le groupe B (variante) avec un taux de ~12 % (0.12)
groupe_b = ____  # np.random.binomial(1, ____, n_b)

df = pd.DataFrame({
    'groupe': ['A'] * n_a + ['B'] * n_b,
    'converti': list(groupe_a) + list(groupe_b),
})
df.head()

## Étape 2 — Statistiques descriptives et visualisation
On calcule le taux de conversion de chaque groupe et on le visualise.

In [ ]:
import matplotlib.pyplot as plt

# TODO : calculez le taux de conversion moyen par groupe (groupby ... ['converti'].mean())
taux = ____
print((taux * 100).round(2))

taux.plot(kind='bar', title='Taux de conversion par groupe')
plt.ylabel('Taux de conversion')
plt.xticks(rotation=0)
plt.show()

## Étape 3 — Test de significativité (test Z sur proportions)
La métrique est un oui/non (converti ou non) : on compare deux proportions avec un **test Z**.
On lit la **p-value** : si elle est < 0,05, l'écart est *statistiquement significatif*.

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

conversions = [groupe_a.sum(), groupe_b.sum()]
observations = [n_a, n_b]

# TODO : appelez proportions_ztest(conversions, observations)
z_stat, p_value = ____
print(f'z-statistique : {z_stat:.3f}')
print(f'p-value       : {p_value:.4f}')

# TODO : complétez la condition de décision (seuil 0,05)
if ____:
    print('=> Différence statistiquement significative (on rejette H0)')
else:
    print("=> Pas de différence significative (on ne rejette pas H0)")

### 🔬 À vous : l'effet de la taille d'échantillon
Relancez l'expérience avec de **petits** groupes (`n = 200`) et observez la p-value.
Objectif : ressentir que moins de données = plus difficile de détecter un vrai écart.

In [ ]:
# TODO : relancez l'expérience avec n = 200 par groupe. La p-value augmente-t-elle ?
np.random.seed(42)
n = 200
a = np.random.binomial(1, 0.10, n)
b = np.random.binomial(1, 0.12, n)
z, p = proportions_ztest([a.sum(), b.sum()], [n, n])
print(f'n={n} -> p-value = {p:.4f}')
# Notez votre observation dans le cahier de TP.

## Étape 4 — Intervalles de confiance à 95 %
Une fourchette plausible du vrai taux de chaque groupe. S'ils ne se chevauchent pas : signe fort d'un écart réel.

In [ ]:
from statsmodels.stats.proportion import proportion_confint

# TODO : calculez l'intervalle de confiance à 95 % pour chaque groupe
ci_a = proportion_confint(groupe_a.sum(), n_a, alpha=0.05)
ci_b = ____  # même chose pour le groupe B
print(f'IC 95% groupe A : [{ci_a[0]*100:.2f}% ; {ci_a[1]*100:.2f}%]')
print(f'IC 95% groupe B : [{ci_b[0]*100:.2f}% ; {ci_b[1]*100:.2f}%]')

## Étape 5 — Exercice en autonomie
Chargez `ab_test_exercice.csv` (écart plus faible : ~10 % vs ~10,5 %) et refaites le test **seuls**.
Concluez : le résultat est-il significatif ? Écrivez votre conclusion en une phrase.

In [ ]:
# Exercice autonome : chargez le CSV et refaites le test SEULS
ex = pd.read_csv('ab_test_exercice.csv')
print(ex.groupby('groupe')['converti'].agg(['sum', 'count', 'mean']))

# TODO : extrayez conversions et effectifs, puis lancez proportions_ztest
conv = ____
obs  = ____
z2, p2 = ____
print(f'z = {z2:.3f}   p-value = {p2:.4f}')
print('Significatif à 5% ?', 'OUI' if p2 < 0.05 else 'NON')

### ✍️ Ma conclusion

*(Écrivez ici votre conclusion en une phrase : « Le résultat est / n'est pas significatif car... »)*

→ 